# 02 — Comparison of all models + Grad-CAM

Run this notebook after the training runs (`01_train_colab.ipynb`). It can be re-run at any time:
it always uses the latest run of every model/strategy found in `MyDrive/road-damage-tl/runs/`.

`Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Outputs (also copied to `MyDrive/road-damage-tl/results/` and downloaded as a zip, to be committed
in `results/` of the repository):
- global results table (paper Section IV.6), ROC curves, confusion matrices, learning curves,
  performance-vs-cost figure;
- Grad-CAM figures and the road-focus ratio of the best strategy of each model (Section V).

In [ ]:
GRADCAM_ALL_RUNS = False   # True: Grad-CAM for every run (slower); False: best strategy of each model

REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"
REPO_DIR = "/content/road-damage-tl"
RAW_DIR = "/content/data/raw"

## 1. Setup

In [ ]:
import glob, json, os, subprocess
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q timm albumentations grad-cam

if not os.path.isdir(f"{RAW_DIR}/sih_road_dataset"):
    zips = sorted(glob.glob(f"{DRIVE_ROOT}/datasets/*.zip"))
    assert zips, "No dataset zip on Drive: run 00_colab_setup.ipynb first."
    os.makedirs(RAW_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", RAW_DIR], check=True)

## 2. Global comparison

In [ ]:
!python -W ignore -m src.compare --runs "{DRIVE_ROOT}/runs" --out results

from IPython.display import Image, Markdown, display
display(Markdown(open("results/global_results.md").read()))
for f in ["accuracy_vs_cost", "val_curves", "roc_best_per_model", "confusion_best_per_model"]:
    print(f)
    display(Image(f"results/{f}.png", width=1000))

## 3. Grad-CAM

In [ ]:
import pandas as pd

runs = []
for f in glob.glob(f"{DRIVE_ROOT}/runs/*/metrics.json"):
    d = os.path.dirname(f)
    model, strategy, stamp = os.path.basename(d).split("__")
    runs.append({"model": model, "strategy": strategy, "stamp": stamp, "dir": d,
                 "f1": json.load(open(f))["test_f1_macro"]})
runs = pd.DataFrame(runs).sort_values("stamp").groupby(["model", "strategy"]).tail(1)
selected = runs if GRADCAM_ALL_RUNS else runs.loc[runs.groupby("model").f1.idxmax()]
print(selected[["model", "strategy", "f1"]].to_string(index=False))

for d in selected.dir:
    print("Grad-CAM:", os.path.basename(d))
    !python -W ignore -m src.explain.gradcam --run "{d}" --data-root {RAW_DIR} --out results/gradcam 2>&1 | grep -E "road_focus_mean|Saved|Traceback|Error"

### Road-focus ratio

Share of the Grad-CAM energy in the lower half of the image (where the road usually is).
Around 0.5 or below means the model also looks at the background (sky, buildings, trees):
a sign of the source bias found in the EDA. It is a proxy, to be interpreted with the figures.

In [ ]:
rows = [json.load(open(f)) for f in sorted(glob.glob("results/gradcam/*/road_focus.json"))]
focus = pd.DataFrame([{"model": r["model"], "strategy": r["strategy"], "road_focus_mean": r["road_focus_mean"],
                       "correct": r["road_focus_correct"], "errors": r["road_focus_errors"],
                       **{f"class_{k}": v for k, v in r["road_focus_per_class"].items()}} for r in rows])
focus.to_csv("results/gradcam/road_focus_summary.csv", index=False)
focus.round(3)

In [ ]:
for d in sorted(glob.glob("results/gradcam/*/")):
    for f in ["correct.png", "errors.png"]:
        if os.path.exists(d + f):
            print(d + f)
            display(Image(d + f, width=1100))

## 4. Save the results (Drive + zip to commit in the repository)

In [ ]:
!mkdir -p "{DRIVE_ROOT}/results" && cp -r results/* "{DRIVE_ROOT}/results/"
!cd /content/road-damage-tl && zip -q -r /content/results.zip results -x "results/eda/*"

from google.colab import files
files.download("/content/results.zip")